In [1]:
import numpy as np
import csv

# Define months and counties
months = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

# Given clean rainfall data for the first 4 counties (mm)
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

# Raw text data for Machakos requiring cleaning
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]

In [2]:
def clean_machakos(raw_data, month_names):
    """
    Cleans raw Machakos rainfall data using try/except ValueError,
    imputes invalid entries with the rounded average of valid values,
    and prints the imputed months.
    """
    valid_vals = []
    parsed_vals = []
    imputed_indices = []

    for i, val in enumerate(raw_data):
        try:
            # Attempt to strip whitespace and convert to integer
            v = int(val.strip())
            parsed_vals.append(v)
            valid_vals.append(v)
        except (ValueError, AttributeError):
            parsed_vals.append(None)
            imputed_indices.append(i)

    # Calculate average of valid values and round it
    avg_val = round(sum(valid_vals) / len(valid_vals))

    # Replace None with the rounded average
    cleaned_vals = []
    for i, v in enumerate(parsed_vals):
        if v is None:
            cleaned_vals.append(avg_val)
            print(f"Imputed month {i+1} ({month_names[i]}) with average value: {avg_val}")
        else:
            cleaned_vals.append(v)

    return cleaned_vals

# Clean Machakos data
machakos_cleaned = clean_machakos(machakos_raw, months)

# Combine all five counties into a single 5 x 12 NumPy array
all_data = clean_rows + [machakos_cleaned]
rainfall_array = np.array(all_data, dtype=int)

print("\n--- Array Confirmation ---")
print(f"Shape: {rainfall_array.shape}")
print(f"Dtype: {rainfall_array.dtype}")

Imputed month 4 (April) with average value: 68
Imputed month 9 (September) with average value: 68

--- Array Confirmation ---
Shape: (5, 12)
Dtype: int64


In [3]:
# 1. Annual total rainfall per county
annual_totals = rainfall_array.sum(axis=1)

# 2. Average rainfall per month across all counties
monthly_averages = rainfall_array.mean(axis=0)

# 3. Wettest month per county using np.argmax
wettest_indices = np.argmax(rainfall_array, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

# 4. Count dry months (< 50 mm) using a boolean mask
dry_mask = rainfall_array < 50
dry_months_counts = dry_mask.sum(axis=1)

# 5. Recommended planting window (two consecutive months with highest combined rainfall)
# Using shifted-array technique: R[:, :-1] + R[:, 1:]
consecutive_sums = rainfall_array[:, :-1] + rainfall_array[:, 1:]
best_window_start_indices = np.argmax(consecutive_sums, axis=1)
planting_windows = [
    f"{months[idx]} - {months[idx+1]}"
    for idx in best_window_start_indices
]

# Display interim results
print("\n--- Analysis Results Summary ---")
for i, county in enumerate(counties):
    print(f"{county}: Annual Total = {annual_totals[i]} mm | Wettest = {wettest_months[i]} | Dry Months = {dry_months_counts[i]} | Planting Window = {planting_windows[i]}")

print("\nMonthly Averages Across All Counties:")
for m, avg in zip(months, monthly_averages):
    print(f"  {m}: {avg:.1f} mm")


--- Analysis Results Summary ---
Nairobi: Annual Total = 1045 mm | Wettest = April | Dry Months = 4 | Planting Window = April - May
Nakuru: Annual Total = 916 mm | Wettest = April | Dry Months = 3 | Planting Window = April - May
Kisumu: Annual Total = 1371 mm | Wettest = April | Dry Months = 0 | Planting Window = April - May
Mombasa: Annual Total = 1075 mm | Wettest = May | Dry Months = 2 | Planting Window = April - May
Machakos: Annual Total = 811 mm | Wettest = November | Dry Months = 6 | Planting Window = November - December

Monthly Averages Across All Counties:
  January: 47.0 mm
  February: 47.6 mm
  March: 91.0 mm
  April: 164.4 mm
  May: 171.2 mm
  June: 67.6 mm
  July: 56.6 mm
  August: 58.2 mm
  September: 64.2 mm
  October: 67.8 mm
  November: 127.4 mm
  December: 80.6 mm


In [4]:
csv_filename = "rainfall_summary.csv"
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

rows = []
for i, county in enumerate(counties):
    rows.append([
        county,
        int(annual_totals[i]),
        wettest_months[i],
        int(dry_months_counts[i]),
        planting_windows[i]
    ])

# Write using the csv module
with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print(f"\nSuccessfully wrote summary to {csv_filename}")


Successfully wrote summary to rainfall_summary.csv


## Planting Advisory
Farmers across all five counties should synchronize their agricultural preparation with the identified bimodal or extended rainy seasons to maximize crop yields. In regions like Nairobi, Nakuru, Kisumu, and Mombasa, planting should commence immediately before or at the start of their peak two-month rainfall windows to ensure adequate soil moisture during germination. For Machakos, where manual data entry errors required imputation, agricultural extension officers must account for historical variability by supplementing natural rainfall with supplemental irrigation if dry spells persist. Monitoring monthly dry-month counts will further help farmers deploy drought-resistant crop varieties during prolonged dry intervals.

## Data Quality Notes
The rainfall dataset combined verified automated station readings with manually transcribed data from Machakos. Hand-typed records in Machakos contained missing entries and string artifacts ("N/A", blank values), which were successfully cleaned using robust try/except error handling and imputed with the rounded mean of valid monthly observations. While imputation preserves dataset dimensions for vectorized NumPy analysis, users should note that imputed values represent statistical estimates rather than true meteorological measurements.